In [6]:
import pytse_client as tse

_cache = {}


def _num(value, kind=float, default=0):
    """تبدیل امن به عدد. اگه None یا غیرقابل‌تبدیل بود، default برمی‌گردونه."""
    if value is None:
        return default
    try:
        return kind(value)
    except (TypeError, ValueError):
        return default


def _str(value, default="—"):
    """تبدیل امن به متن."""
    if value is None:
        return default
    return str(value).strip()


def _day(value):
    text = str(value).split(".")[0]
    return text[:10] if len(text) >= 10 and text[4] == "-" else text


def load(symbol):
    """داده کامل یه نماد رو از tsetmc می‌گیره و دیکشنری برمی‌گردونه."""
    if symbol in _cache:
        return _cache[symbol]

    t = tse.Ticker(symbol)

    history = [
        {
            "date": _day(r["date"]),
            "adj": _num(r["adjClose"], int),
            "last": _num(r["close"], int),
            "volume": _num(r["volume"], int),
        }
        for r in t.history.to_dict("records")
    ]

    clients = [
        {
            "date": _day(r["date"]),
            "real": _num(r["individual_buy_vol"], int),
            "legal": _num(r["corporate_buy_vol"], int),
        }
        for r in t.client_types.to_dict("records")
    ]
    clients.reverse()

    data = {
        "title": _str(t.title),
        "group": _str(t.group_name),
        "market": _str(t.flow),
        "last": _num(t.last_price, int),
        "yday": _num(t.yesterday_price, int),
        "pe": _num(t.p_e_ratio, float),
        "history": history,
        "clients": clients,
    }
    _cache[symbol] = data
    return data

In [9]:
import requests, time
t = time.time()
r = requests.get("https://cdn.tsetmc.com/api/ClosingPrice/GetMarketWatch", 
                 headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
print(f"زمان: {time.time()-t:.1f}s | وضعیت: {r.status_code}")

زمان: 0.5s | وضعیت: 200


In [8]:
import requests, time
t = time.time()
r = requests.get("https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceInfo/46348559193224090",
                 headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
print(f"زمان: {time.time()-t:.1f}s")

زمان: 0.5s


In [7]:
import time
from load_data import load

t = time.time()
print("شروع...")
d = load("فولاد")
print(f"تموم شد در {time.time()-t:.1f} ثانیه")
print(f"تعداد روز تاریخچه: {len(d['history'])}")

ModuleNotFoundError: No module named 'load_data'